In [ ]:
from pathlib import Path
import json
import pandas as pd

root = Path("../../data/raw/Training/TL_보험/")
rows = []
errors = []

for path in root.rglob("*.json"):
    try:
        with path.open("r", encoding="utf-8") as f:
            item = json.load(f)

        # 출처 정보도 보존하면 나중에 추적하기 좋음
        item["_source_file"] = str(path)

        rows.append(item)

    except Exception as e:
        errors.append((str(path), str(e)))

df = pd.json_normalize(rows)

print(df.shape)
print(f"읽기 실패: {len(errors)}개")

(24000, 14)
읽기 실패: 0개


In [2]:
df.head(1).qa_data[0]

[{'qa_id': '21-1_ins_01_000009_001',
  'task_category': '핵심 금융 용어',
  'consulting_situation': '일반문의',
  'qa_topic': '자동차보험가입/해지/변경등',
  'consulting_purpose': '마일리지 환급 안내',
  'core_financial_terms': '마일리지',
  'input_length': 291,
  'instruction': '마일리지 환급 절차와 연말 환급 일정을 고객에게 안내하시오.',
  'input': {'question': '연말에 마일리지 환급이 진행된다고 들었는데, 그 절차가 구체적으로 어떻게 이루어지는지 자세하게 설명해 주실 수 있나요? 환급 과정에서 어떤 단계들이 포함되는지 궁금합니다.',
   'answer': '연말 환급 절차는 먼저 고객님의 차량 번호와 소유자 정보를 정확히 확인하는 단계부터 시작합니다. 이후 고객님의 마일리지 사용 내역을 꼼꼼히 조회하고, 환급을 위해 필요한 서류와 차량 변경 후 마일리지 승계가 확인된 사진을 제출해 주시면 담당 부서에서 이를 검토하게 됩니다. 모든 서류와 사진이 정상적으로 접수되면 해당 월 내에 환급이 이루어지도록 처리됩니다.',
   'follow_up_question': '그러면 연말 환급이 정확히 어느 시점에 진행되는지 알려 주시고, 환급을 위해 꼭 준비해야 하는 서류 목록과 구체적인 일정도 함께 안내해 주실 수 있을까요?'},
  'output': '연말 마일리지 환급 절차는 먼저 고객님의 차량 번호와 소유자 성함을 확인하고, 현재 보유하고 계신 마일리지 사용 내역을 조회합니다. 이후 차량 변경으로 인한 마일리지 승계가 필요한 경우, 승계 사진과 함께 필요한 서류를 제출해 주셔야 합니다. 제출된 서류와 사진이 정상적으로 접수되면 검토가 진행되며, 검토가 완료된 후에는 해당 연말 월에 환급이 이루어집니다. 환급 일정은 보통 연말 정산이 완료되는 시점에 맞추어 진행되며, 정확한 일자는 서류

In [ ]:
# extract 'qa_topic'

df["qa_data"].explode().apply(
    lambda x: x.get("qa_topic") if isinstance(x, dict) else None
).value_counts()

qa_data
운전자/질병/상해보험등계약내용변경/해지    9204
자동차보험가입/해지/변경등           5324
기타계약관련문의                 4335
운전자/질병/상해보험등보험금청구/확인     2633
자동차사고접수                  2504
Name: count, dtype: int64

In [ ]:
#consulting_purpose
a= df["qa_data"].explode().apply(
    lambda x: x.get("consulting_purpose") if isinstance(x, dict) else None
).unique()

print(len(a))

8727


In [ ]:
#core_financial_terms
a= df["qa_data"].explode().apply(
    lambda x: x.get("core_financial_terms") if isinstance(x, dict) else None
).value_counts()

In [28]:
mask = df["qa_data"].apply(
    lambda qa_list: any(
        isinstance(qa, dict)
        and qa.get("core_financial_terms") == "요양급여"
        for qa in (qa_list if isinstance(qa_list, list) else [])
    )
)

result = df.loc[mask].copy()
result

,qa_data,_source_file,source.source_institution,source.source_id,source.source_date,source.client_gender,source.client_age,source.consulting_client,source.consulting_client_type,source.source_length,source.consulting_content,consulting.consulting_category,consulting.consulting_topic,consulting.consulting_summary
14199,"[{'qa_id': '21-1_ins_03_015107_001', 'task_cat...",data\Training\TL_보험\03\21-1_ins_03_015107_001....,하나손해보험,21-1_ins_03_015107,202503,여,50~59세,기존고객,None,1252,"TX 반갑습니다. ●●●입니다.\nRX 안녕하세요, 제가 현재 가입한 간병보험에 대...",보험,계약내용변경/해지,"고객은 간병보험의 요양 급여 지급 기준과 연간 일회 지급 외 추가 청구 가능 항목,..."


# EDA 

1. UNIQUE 한 상담 주제는 어떻게 이루어져 있는가?
2. 그 개수는 어떻게 이루어져 있는가

# 현재 필요한 데이터 목록

consulting.consulting_category : 상담 카테고리 

consulting.consulting_topic : 세부 상담 주제

qa_data[0]['qa_topic'] : 상세 주제

qa_data[0]['consulting_purpose'] : 목적

qa_data[0]['instruction'] : 학습용 구조 - 요구 요약문

qa_data[0]['input']['question'] : 학습용 인풋 - 사용자 질문
 
qa_data[0]['input']['answer'] : 학습용 답안 - 답안

qa_data[0]['input']['follow_up_question'] : 학습용 추가 질문 - 또 어떤걸 준비해야 하나요

qa_data[0]['output'] : 종합 답안

RAG[META] : 
- 상담 카테고리(consulting_category) 
- 세부 상담 주제(consulting_topic) 
- QA_TOPIC 
- Consulting_purpose


RAG[TEXT] : 요구사항 / 고객 질문 / 상담사 답변 / 꼬리질문 / 종합 답변 

LLM FT : 고객질문 - 상담사 답변 // 고객질문+꼬리질문 - 최종 답안 

inst :
- 고객질문 - 상담사 답변 
- 고객질문+꼬리질문 - 최종 답안 

input
- 고객질문+꼬리질문

output
- 상담사 답변
- 최종 답안

In [ ]:
RAG_LABELS = {
    "instruction": "요구사항",
    "question": "고객 질문",
    "answer": "상담사 답변",
    "follow_up_question": "꼬리 질문",
    "output": "종합 답변",
}


def normalize_text(value):
    return str(value or "").replace("\u2028", "\n").replace("\u2029", "\n\n").strip()


def build_rag_row(item, qa):
    consulting = item.get("consulting") or {}
    qa_input = qa.get("input") or {}
    values = {
        "instruction": qa.get("instruction"),
        "question": qa_input.get("question"),
        "answer": qa_input.get("answer"),
        "follow_up_question": qa_input.get("follow_up_question"),
        "output": qa.get("output"),
    }
    return {
        "consulting_category": consulting.get("consulting_category"),
        "consulting_topic": consulting.get("consulting_topic"),
        "qa_topic": qa.get("qa_topic"),
        "consulting_purpose": qa.get("consulting_purpose"),
        "text": "\n".join(
            f"{RAG_LABELS[key]}: {normalize_text(value)}"
            for key, value in values.items()
        ),
    }


rag_rows = []
rag_errors = []
for path in Path("../../data/raw/Training").rglob("*.json"):
    try:
        with path.open(encoding="utf-8") as file:
            item = json.load(file)
        rag_rows.extend(
            build_rag_row(item, qa)
            for qa in (item.get("qa_data") or [])
            if isinstance(qa, dict)
        )
    except (OSError, json.JSONDecodeError, TypeError) as error:
        rag_errors.append({"file": str(path), "error": str(error)})

rag_df = pd.DataFrame(rag_rows)
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)
rag_df.to_csv(output_dir / "rag_dataset.csv", index=False, encoding="utf-8-sig")
rag_df.to_json(
    output_dir / "rag_dataset.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

print(f"RAG 문서: {len(rag_df):,}개 / 읽기 실패: {len(rag_errors):,}개")
rag_df.head()

In [ ]:
FT_COLUMNS = ["instruction", "input", "output"]


def build_ft_rows(item):
    if not isinstance(item, dict):
        raise TypeError("최상위 JSON은 객체여야 합니다.")

    v1_rows, v2_rows = [], []
    for qa in item.get("qa_data") or []:
        if not isinstance(qa, dict):
            continue

        qa_input = qa.get("input") or {}
        instruction = normalize_text(qa.get("instruction"))
        question = normalize_text(qa_input.get("question"))
        answer = normalize_text(qa_input.get("answer"))
        follow_up = normalize_text(qa_input.get("follow_up_question"))
        final_answer = normalize_text(qa.get("output"))

        if not all((instruction, question, answer, follow_up, final_answer)):
            raise ValueError(f"필수 FT 필드 누락: {qa.get('qa_id', 'unknown')}")

        v1_rows.append({
            "instruction": instruction,
            "input": question,
            "output": final_answer,
        })
        v2_rows.extend([
            {"instruction": instruction, "input": question, "output": answer},
            {
                "instruction": instruction,
                "input": f"고객 질문: {question}\n꼬리 질문: {follow_up}",
                "output": final_answer,
            },
        ])

    return v1_rows, v2_rows


ft_summary = []
for split in ("Training", "Validation"):
    ft_v1_rows, ft_v2_rows, ft_errors = [], [], []
    for path in Path("data", split).rglob("*.json"):
        try:
            with path.open(encoding="utf-8") as file:
                item = json.load(file)
            file_v1_rows, file_v2_rows = build_ft_rows(item)
            ft_v1_rows.extend(file_v1_rows)
            ft_v2_rows.extend(file_v2_rows)
        except (OSError, json.JSONDecodeError, TypeError, ValueError) as error:
            ft_errors.append({"file": str(path), "error": str(error)})

    ft_v1_df = pd.DataFrame(ft_v1_rows, columns=FT_COLUMNS)
    ft_v2_df = pd.DataFrame(ft_v2_rows, columns=FT_COLUMNS)
    split_name = split.lower()
    ft_v1_df.to_json(
        output_dir / f"ft_v1_{split_name}.jsonl",
        orient="records", lines=True, force_ascii=False,
    )
    ft_v2_df.to_json(
        output_dir / f"ft_v2_{split_name}.jsonl",
        orient="records", lines=True, force_ascii=False,
    )
    ft_summary.append({
        "split": split,
        "v1_rows": len(ft_v1_df),
        "v2_rows": len(ft_v2_df),
        "errors": len(ft_errors),
    })

pd.DataFrame(ft_summary)

## text의 구성

각 항목을 지정해서 텍스트로 넣기

메타 데이터들은 컬럼으로 따로 분류해서

consulting_category
consulting_topic
qa_topic
consulting_purpose